# rPPG error analysis: open coding template

Hamel-style error analysis for the eval kit. Work through the worst sessions one at a time,
write what went wrong in your own words, then group those notes into failure modes.

1. Run the evaluation and export the worst sessions:

   ```bash
   cd service-intelligence
   python -m learning.eval_kit.eval_runner --manifest MANIFEST.json --output-dir out/eval
   python -m learning.eval_kit.error_analysis --eval-dir out/eval --manifest MANIFEST.json \
       --method pbv_i30 --top 30
   ```

2. Set `EVAL_DIR` below and run the cells. For each session, look at the plot and fill in
   `failure_mode`, `root_cause`, `binary_pass` (pass/fail as a product decision) and
   `fix_hypothesis` in `open_coding.csv`. Save as a new file per reviewer.
3. Pivot the codes to find the failure modes worth fixing first. With two reviewers, check
   agreement on `binary_pass` with Cohen's kappa before trusting the counts.

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

EVAL_DIR = Path("out/eval")
EA_DIR = EVAL_DIR / "error_analysis"
CODING_FIELDS = ["failure_mode", "root_cause", "binary_pass", "fix_hypothesis"]

coding = pd.read_csv(EA_DIR / "open_coding.csv", keep_default_na=False)
coding[["rank", "session_id", "failure_kind", "hr_pred", "hr_ref", "abs_err", "conf",
        "fitz_group", "fps_bin", "lighting", "motion"]]

## Review one session at a time

Change `RANK` and re-run. Write notes before reading any automated flags, so the flags don't
steer what you see. Suggested starting codes (add your own): `harmonic_lock`,
`motion_artifact`, `low_light_noise`, `exposure_step`, `flicker_alias`, `low_fps_or_jitter`,
`weak_pulse_darker_skin`, `false_accept_no_pulse`, `over_abstention`, `reference_problem`.

In [ ]:
RANK = 1
row = coding.loc[coding["rank"] == RANK].iloc[0]
print(row[["session_id", "failure_kind", "hr_pred", "hr_raw", "hr_ref", "conf",
           "window_abstain_rate", "window_mae", "flags"]].to_string())
if row["plot"]:
    display(Image(filename=str(EA_DIR / row["plot"])))

In [ ]:
# Record codes here, or edit the CSV in a spreadsheet and reload it.
coding.loc[coding["rank"] == RANK, CODING_FIELDS] = [
    "",  # failure_mode, e.g. "harmonic_lock"
    "",  # root_cause, e.g. "dicrotic notch stronger than fundamental at low AC"
    "",  # binary_pass: "pass" or "fail"
    "",  # fix_hypothesis, e.g. "SHS with 4 harmonics; prior on previous window"
]
coding.to_csv(EA_DIR / "open_coding_reviewer1.csv", index=False)

## Axial coding: which failure modes matter most

In [ ]:
coded = coding[coding["failure_mode"] != ""]
print(f"Coded {len(coded)} of {len(coding)} sessions")
summary = (
    coded.groupby("failure_mode")
    .agg(n=("session_id", "count"), mean_err=("abs_err", lambda s: pd.to_numeric(s, errors="coerce").mean()))
    .sort_values("n", ascending=False)
)
summary

In [ ]:
# Where do the failure modes sit? Look for modes that pile up in one Fitzpatrick group.
pd.crosstab(coded["failure_mode"], coded["fitz_group"], margins=True)

In [ ]:
pd.crosstab(coded["failure_mode"], [coded["lighting"], coded["motion"]])

## Agreement between two reviewers

Both reviewers code the same sessions independently. Kappa below about 0.6 means the
pass/fail rule is not yet clear enough to count failures with.

In [ ]:
def cohen_kappa(a, b):
    a, b = pd.Series(a, dtype=str), pd.Series(b, dtype=str)
    labels = sorted(set(a) | set(b))
    observed = (a.values == b.values).mean()
    expected = sum((a == lab).mean() * (b == lab).mean() for lab in labels)
    return (observed - expected) / (1 - expected) if expected < 1 else 1.0


r1_path, r2_path = EA_DIR / "open_coding_reviewer1.csv", EA_DIR / "open_coding_reviewer2.csv"
if r1_path.exists() and r2_path.exists():
    r1 = pd.read_csv(r1_path, keep_default_na=False).set_index("session_id")
    r2 = pd.read_csv(r2_path, keep_default_na=False).set_index("session_id")
    both = r1.join(r2, lsuffix="_r1", rsuffix="_r2", how="inner")
    both = both[(both["binary_pass_r1"] != "") & (both["binary_pass_r2"] != "")]
    print(f"kappa on binary_pass over {len(both)} sessions:",
          round(cohen_kappa(both["binary_pass_r1"], both["binary_pass_r2"]), 3))
else:
    print("Add open_coding_reviewer2.csv to compute agreement.")

## Scorecard context

The release gate and the stratified tables are in `scorecard.md`. Read the abstention-rate
ratio next to the parity gap: a model that lowers darker-skin error by refusing those users is
not fixed.

In [ ]:
print((EVAL_DIR / "scorecard.md").read_text()[:3000])